# AKCMamba-YOLO nguyên bản — kiểm tra cấu hình tác giả
**Chưa có cấu hình kiến trúc AKCMamba chạy được trong source được công bố.**

Repo này đã khôi phục source từ ZIP gốc, không còn hybrid YOLOv12s. Kiểm tra repo tác giả tại commit `6bfbfc7b9c69c431f85eae3f6b187402f52557e3`: ZIP giống hệt bản người dùng cung cấp. Không tìm thấy YAML sử dụng module AKCMamba. YAML yolov8.yaml chỉ là YOLOv8 thường.

Không chọn yolov8s và gọi kết quả là AKCMamba. Không có đủ căn cứ để tự gán AKC-S/T tương đương YOLOv12s. Notebook này clone source và kiểm tra điều kiện, **chưa thể train AKCMamba** cho tới khi tác giả cung cấp cấu hình và module đầy đủ.


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working/AKCMamba_YOLO')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/mluu59990-collab/AKCMamba_YOLO.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['git','rev-parse','HEAD'],check=True)


## Kiểm kê YAML và các lớp AKC/SSM trong source
Chỉ đọc source, không tự sửa kiến trúc.

In [ ]:
import ast
model_root=ROOT/'ultralytics/cfg/models'
yamls=sorted(model_root.rglob('*.yaml'))
print('Model YAMLs:',len(yamls))
candidates=[]
for path in yamls:
    text=path.read_text(errors='replace')
    if any(token in text for token in ('AKCMamba','AKC3','AKC4','AKSS2D','3CAKC','4CAKC')):
        candidates.append(path)
print('YAML candidates for AKCMamba:',[str(p.relative_to(ROOT)) for p in candidates])
for path in (ROOT/'ultralytics/nn/modules').rglob('*.py'):
    try: tree=ast.parse(path.read_text(errors='replace'))
    except (SyntaxError,UnicodeError): continue
    classes=[n.name for n in ast.walk(tree) if isinstance(n,ast.ClassDef) and any(t in n.name for t in ('AKC','SS2D','VSSBlock','XSSBlock'))]
    if classes: print(path.relative_to(ROOT),classes)
if not candidates:
    raise RuntimeError('SOURCE THIEU CAU HINH AKCMamba: can YAML kien truc va cac module day du tu tac gia. Khong train yolov8.yaml thay the.')
raise RuntimeError('Co YAML ung vien moi: can doi chieu module/parser va paper truoc khi chon ban nho. Notebook chua xac minh source moi nay.')


## Cần bổ sung để train dữ liệu của bạn
- YAML kiến trúc AKCMamba đầy đủ từ tác giả, cùng lựa chọn scale nếu có.
- Implementation tương ứng của 3CAKCMamba/4CAKCMamba và CUDA selective scan cùng dependency.

Sau khi có source đầy đủ mới có thể tạo cell cài đặt, chọn phiên bản nhỏ và train đúng model. Việc thiếu source không được giải quyết bằng cách đổi tên YOLOv8s thành AKCMamba.
